In [ ]:
import os, pyspark.sql.functions as F
from pyspark.sql import SparkSession

pg_url = f"jdbc:postgresql://{os.getenv('PG_HOST','postgres')}:{os.getenv('PG_PORT','5432')}/{os.getenv('PG_DB','nyc_taxi')}"
pg_props = {"user": os.getenv("PG_USER","postgres"),
            "password": os.getenv("PG_PASSWORD","postgres"),
            "driver":"org.postgresql.Driver"}

spark = (SparkSession.builder
         .appName("NYC TLC Ingesta RAW")
         .config("spark.jars.packages","org.postgresql:postgresql:42.7.4")
         .getOrCreate())

RUN_ID = os.getenv("RUN_ID","dev_001")
print("Spark OK | PG_URL:", pg_url, "| RUN_ID:", RUN_ID)

In [ ]:
def write_jdbc(df, table):
    (df
     .withColumn("run_id", F.lit(RUN_ID))
     .withColumn("ingested_at_utc", F.current_timestamp())
     .write
     .mode("append")
     .jdbc(pg_url, table, properties=pg_props))

In [ ]:
# EDITA estas rutas a donde tengas tus Parquets locales
BASE = "/data/taxi"  # ejemplo
# Estructura esperada:
# /data/taxi/yellow/2015-01.parquet, ...  /data/taxi/green/2015-01.parquet
services = ["yellow","green"]
years = list(range(2015, 2026))
months = list(range(1,13))

In [ ]:
import os

for svc in services:
    for y in years:
        for m in months:
            path = f"{BASE}/{svc}/{y}-{m:02d}.parquet"
            if not os.path.exists(path):
                print(f"[SKIP] {path} no existe")
                continue
            df = spark.read.parquet(path)
            if svc == "yellow":
                df = (df
                      .withColumnRenamed("tpep_pickup_datetime","pickup_datetime")
                      .withColumnRenamed("tpep_dropoff_datetime","dropoff_datetime")
                      .withColumnRenamed("PULocationID","pu_location_id")
                      .withColumnRenamed("DOLocationID","do_location_id"))
                table = "raw.yellow_taxi_trip"
            else:
                df = (df
                      .withColumnRenamed("lpep_pickup_datetime","pickup_datetime")
                      .withColumnRenamed("lpep_dropoff_datetime","dropoff_datetime")
                      .withColumnRenamed("PULocationID","pu_location_id")
                      .withColumnRenamed("DOLocationID","do_location_id"))
                table = "raw.green_taxi_trip"

            df = (df
                  .withColumn("source_year", F.lit(y))
                  .withColumn("source_month", F.lit(m))
                  .withColumn("service_type", F.lit(svc)))
            cnt = df.count()
            print(f"[INGEST] {svc} {y}-{m:02d} rows={cnt} -> {table}")
            write_jdbc(df, table)

In [ ]:
# EDITA la ruta del CSV de zones
zones_csv = f"{BASE}/taxi_zone_lookup.csv"
zones = spark.read.csv(zones_csv, header=True, inferSchema=True)
# Estandariza nombre de columna
zones = zones.withColumnRenamed("LocationID","LocationID")
zones = zones.withColumnRenamed("Borough","borough").withColumnRenamed("Zone","zone")
zones = zones.selectExpr("LocationID as locationid","borough","zone","service_zone")
write_jdbc(zones, "raw.taxi_zone_lookup")
print("[DONE] taxi_zone_lookup")

In [ ]:
for t in ["raw.yellow_taxi_trip", "raw.green_taxi_trip", "raw.taxi_zone_lookup"]:
    df = (spark.read
          .format("jdbc")
          .option("url", pg_url)
          .option("dbtable", t)
          .option("user", pg_props["user"])
          .option("password", pg_props["password"])
          .option("driver", pg_props["driver"])
          .load())
    print(t, "->", df.count(), "filas")

In [ ]:
for t in ["raw.yellow_taxi_trip", "raw.green_taxi_trip", "raw.taxi_zone_lookup"]:
    df = (spark.read
          .format("jdbc")
          .option("url", pg_url)
          .option("dbtable", t)
          .option("user", pg_props["user"])
          .option("password", pg_props["password"])
          .option("driver", pg_props["driver"])
          .load())
    print(t, "->", df.count(), "filas")